https://colab.research.google.com/drive/1t4BY0WfXfTYn3rUuhjqLdJhgbgb-82_E?authuser=2


In [ ]:
# Install required packages
!pip install -q torch>=2.0.0
!pip install -q transformers>=4.30.0
!pip install -q datasets>=2.0.0
!pip install -q accelerate>=0.20.0
!pip install -q evaluate>=0.4.0
!pip install -q seqeval>=1.2.2
!pip install -q scikit-learn>=1.3.0
!pip install -q pandas>=2.0.0
!pip install -q matplotlib>=3.7.0
!pip install -q simple-icd-10>=0.1.0
!pip install -q gradio>=4.0.0

# Ensure output directories exist for plots and other assets
import os
output_dir_plots = "./demo_outputs/plots"
output_dir_summaries = "./demo_outputs/summaries"

os.makedirs(output_dir_plots, exist_ok=True)
os.makedirs(output_dir_summaries, exist_ok=True)
print(f"Output directories created/ensured: {output_dir_plots}, {output_dir_summaries}")


Output directories created/ensured: ./demo_outputs/plots, ./demo_outputs/summaries


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSeq2SeqLM
)
from datasets import load_dataset, Dataset
import evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import json
import difflib
import re
from typing import List, Dict, Tuple
import warnings
import os
import logging
from pathlib import Path

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# NEW: Centralized Configuration Dictionary - UPDATED FOR NEW JSON FILES
config = {
    "ner_model_name": "d4data/biomedical-ner-all",
    "summarizer_model_name": "facebook/bart-large-cnn",
    "max_seq_length": 512,
    "ner_confidence_threshold": 0.80,
    "icd10_direct_codes_file": "icd_mapping.json",
    "critical_patterns_file": "critical_condition.json",
    "summarization_min_length": 50,
    "summarization_max_length": 150,
    "summarization_do_sample": False,
    "summarization_top_k": 50,
    "summarization_top_p": 0.95,
    "summarization_temperature": 0.7,
    "plot_output_dir": "./demo_outputs/plots",
    "summary_output_dir": "./demo_outputs/summaries"
}
logger.info("Configuration loaded successfully.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"  GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
logger.info("\n" + "="*80)
logger.info("LOADING CLINICAL NOTES DATASET")
logger.info("="*80)

# Define the preferred column name for clinical text
# Based on the error, 'full_note' seems like the best candidate.
CLINICAL_TEXT_COLUMN = 'full_note'
# You could also consider 'note' or 'conversation' depending on content,
# but 'full_note' implies it's the complete clinical entry.

try:
    dataset = load_dataset("AGBonnet/augmented-clinical-notes", split="train")
    logger.info(f"Loaded dataset: {len(dataset)} clinical notes")

    logger.info(f"Dataset columns found: {dataset.column_names}")

    # Check if the chosen column exists
    if CLINICAL_TEXT_COLUMN in dataset.column_names:
        logger.info(f"Using '{CLINICAL_TEXT_COLUMN}' column for clinical text.")
        sample_text = dataset[0][CLINICAL_TEXT_COLUMN][:500] # Use the correct column name
        logger.info("\nSample Clinical Note:")
        logger.info("-" * 80)
        logger.info(sample_text + "...")
        logger.info("-" * 80)
    else:
        error_msg = f"Dataset does not contain the expected '{CLINICAL_TEXT_COLUMN}' column. Available columns: {dataset.column_names}"
        logger.error(error_msg)
        raise KeyError(error_msg)

    # Now, generate note_lengths using the correct column name
    note_lengths = [len(note[CLINICAL_TEXT_COLUMN].split()) for note in dataset] # Use the correct column name
    avg_len = np.mean(note_lengths)
    min_len = np.min(note_lengths)
    max_len = np.max(note_lengths)
    logger.info(f"\nDataset Statistics:")
    logger.info(f"  Total Notes: {len(dataset):,}")
    logger.info(f"  Average Note Length: {avg_len:.0f} words")
    logger.info(f"  Min Note Length: {min_len} words")
    logger.info(f"  Max Note Length: {max_len} words")

    plt.figure(figsize=(10, 6))
    plt.hist(note_lengths, bins=50, color='skyblue', edgecolor='black')
    plt.title('Distribution of Clinical Note Lengths (Words)', fontsize=14)
    plt.xlabel('Number of Words', fontsize=12)
    plt.ylabel('Number of Notes', fontsize=12)
    plt.grid(axis='y', alpha=0.75)
    plt.tight_layout()
    plot_path = os.path.join(config["plot_output_dir"], "note_length_distribution.png")
    plt.savefig(plot_path)
    logger.info(f"Saved note length distribution plot to {plot_path}")
    plt.close()

except KeyError as ke:
    logger.error(f"Dataset column error: {ke}")
    logger.info("Falling back to synthetic medical notes due to missing expected column.")
    synthetic_notes = [
        "Patient with diabetes mellitus type 2 presents for follow-up. Blood glucose well controlled on metformin. HbA1c 6.8%. Continue current regimen.",
        "67-year-old male with hypertension and hyperlipidemia. BP 142/88. Started on lisinopril 10mg daily and atorvastatin 20mg nightly.",
        "Patient presents with acute chest pain. EKG shows ST elevation. Troponin elevated. Diagnosis: acute myocardial infarction. Started on aspirin and heparin.",
        "45-year-old female with asthma exacerbation. Wheezing on exam. Given albuterol nebulizer treatment. Prescribed prednisone 40mg daily for 5 days.",
        "Patient with chronic kidney disease stage 3. Creatinine 1.8. eGFR 45. Continue ACE inhibitor. Nephrology referral placed.",
    ] * 200

    dataset = Dataset.from_dict({"full_note": synthetic_notes}) # Use 'full_note' in synthetic dataset too for consistency
    logger.info(f"Created synthetic dataset with {len(dataset)} notes as fallback using '{CLINICAL_TEXT_COLUMN}' column.")

except Exception as e:
    logger.error(f"An unexpected error occurred during dataset loading: {e}")
    logger.error(f"Error type: {type(e)}")
    logger.info("Falling back to synthetic medical notes due to general error during loading.")
    synthetic_notes = [
        "Patient with diabetes mellitus type 2 presents for follow-up. Blood glucose well controlled on metformin. HbA1c 6.8%. Continue current regimen.",
        "67-year-old male with hypertension and hyperlipidemia. BP 142/88. Started on lisinopril 10mg daily and atorvastatin 20mg nightly.",
        "Patient presents with acute chest pain. EKG showed ST elevation. Troponin elevated. Diagnosis: acute myocardial infarction. Started on aspirin and heparin.",
        "45-year-old female with asthma exacerbation. Wheezing on exam. Given albuterol nebulizer treatment. Prescribed prednisone 40mg daily for 5 days.",
        "Patient with chronic kidney disease stage 3. Creatinine 1.8. eGFR 45. Continue ACE inhibitor. Nephrology referral placed.",
    ] * 200

    dataset = Dataset.from_dict({"full_note": synthetic_notes}) # Use 'full_note' for consistency
    logger.info(f"Created synthetic dataset with {len(dataset)} notes as fallback using '{CLINICAL_TEXT_COLUMN}' column.")


In [ ]:
logger.info("\n" + "="*80)
logger.info("LOADING BIOCLINICALBERT FOR MEDICAL NER")
logger.info("="*80)

# Load model and tokenizer from config
ner_model_name = config["ner_model_name"]
ner_tokenizer = AutoTokenizer.from_pretrained(ner_model_name)
ner_model = AutoModelForTokenClassification.from_pretrained(ner_model_name)
ner_model.to(device) # Move model to GPU

logger.info(f"Loaded NER Model: {ner_model_name} and Tokenizer")
logger.info(f"NER Model on device: {device}")
logger.info(f"Max sequence length for NER: {config['max_seq_length']}")

# Define custom label mapping for biomedical-ner-all model
# This model uses its own specific labels, which need to be mapped to a more general set
# for consistent entity types across the system.
_label_map = {
    "O": "O",
    "B-MEDICATION": "B-MEDICATION", "I-MEDICATION": "I-MEDICATION",
    "B-SYMPTOM": "B-CONDITION", "I-SYMPTOM": "I-CONDITION",
    "B-DIAGNOSIS": "B-CONDITION", "I-DIAGNOSIS": "I-CONDITION",
    "B-PROCEDURE": "B-PROCEDURE", "I-PROCEDURE": "I-PROCEDURE",
    "B-ANATOMY": "B-ANATOMY", "I-ANATOMY": "I-ANATOMY",
    "B-TEST": "B-PROCEDURE", "I-TEST": "I-PROCEDURE", # Mapping TEST to PROCEDURE for simplicity
    "B-BEHAVIOR": "B-CONDITION", "I-BEHAVIOR": "I-CONDITION", # Mapping BEHAVIOR to CONDITION
    # Add other mappings if the base model has more labels
}

# The actual set of labels that will be used in our system (simplified for demo)
# This id2label is for internal representation after mapping
id2label_system = {
    0: 'O',
    1: 'B-CONDITION', 2: 'I-CONDITION',
    3: 'B-MEDICATION', 4: 'I-MEDICATION',
    5: 'B-PROCEDURE', 6: 'I-PROCEDURE',
    7: 'B-ANATOMY', 8: 'I-ANATOMY'
}
label2id_system = {v: k for k, v in id2label_system.items()}

logger.info(f"Defined custom label mapping for {ner_model_name} to system labels.")


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [ ]:
logger.info("\n" + "="*80)
logger.info("DEFINING ENTITY EXTRACTION HELPERS (ROBUST MAPPING)")
logger.info("="*80)

def merge_overlapping_entities(entities: List[Dict], text: str) -> List[Dict]:
    """
    Merges overlapping OR adjacent entities.
    """
    if not entities:
        return []

    # Sort by start position
    entities.sort(key=lambda x: (x['start_char'], -x['end_char']))

    merged_entities = []
    current_entity = entities[0]

    for i in range(1, len(entities)):
        next_entity = entities[i]

        # Check overlap or adjacency (>= handles adjacent subwords)
        if current_entity['end_char'] >= next_entity['start_char']:
            if current_entity['entity_type'] == next_entity['entity_type']:
                # Merge: Extend current_entity
                current_entity['end_char'] = max(current_entity['end_char'], next_entity['end_char'])
                # Reconstruct word from original text
                current_entity['word'] = text[current_entity['start_char']:current_entity['end_char']]
                current_entity['confidence'] = max(current_entity['confidence'], next_entity['confidence'])
            else:
                # Conflict: Keep higher confidence
                if next_entity['confidence'] > current_entity['confidence']:
                    current_entity = next_entity
        else:
            merged_entities.append(current_entity)
            current_entity = next_entity

    merged_entities.append(current_entity)
    return merged_entities

def map_entity_types(entity: Dict) -> Dict:
    """
    Maps raw model labels to system schema (CONDITION, MEDICATION, PROCEDURE, ANATOMY).
    Critically updated to handle 'DISEASE_DISORDER' and 'HISTORY'.
    """
    original_type = entity['entity_type'].upper()

    # --- MAPPING LOGIC START ---
    if any(x in original_type for x in ["MEDICATION", "DRUG", "CHEMICAL"]):
        sys_type = "MEDICATION"

    # Map all disease/symptom variants to CONDITION
    elif any(x in original_type for x in ["DISEASE", "DISORDER", "SYMPTOM", "SIGN", "DIAGNOSIS", "HISTORY", "PROBLEM"]):
        sys_type = "CONDITION"

    # Map procedures and tests
    elif any(x in original_type for x in ["PROCEDURE", "TEST", "CLINICAL_EVENT", "THERAPEUTIC"]):
        sys_type = "PROCEDURE"

    # Map anatomy
    elif any(x in original_type for x in ["ANATOMY", "ORGAN", "BIOLOGICAL"]):
        sys_type = "ANATOMY"

    # Fallback for others (Keep original or ignore?)
    # For extraction purposes, we keep them so we can see what the model found
    else:
        sys_type = original_type
    # --- MAPPING LOGIC END ---

    entity['entity_type'] = sys_type
    return entity

def extract_entities(text, model, tokenizer, confidence_threshold=0.9):
    """
    Extracts entities from text using a token classification model.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=config['max_seq_length'],
        padding=True,
        return_offsets_mapping=True
    ).to(device)

    offset_mapping = inputs.pop('offset_mapping').squeeze().tolist()
    input_ids = inputs['input_ids'].squeeze().tolist()

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits.squeeze()
        probabilities = torch.softmax(logits, dim=-1)

    predictions = torch.argmax(probabilities, dim=-1).tolist()

    entities = []
    current_entity = {"word_ids": [], "entity_type": None, "start_char": -1, "end_char": -1, "confidence": 0.0}
    all_token_confidences = []

    for i, (pred_id, input_id, offset) in enumerate(zip(predictions, input_ids, offset_mapping)):
        token = tokenizer.convert_ids_to_tokens(input_id)

        if token in tokenizer.all_special_tokens or offset == (0, 0):
            continue

        label = model.config.id2label[pred_id]
        token_confidence = probabilities[i][pred_id].item()
        all_token_confidences.append(token_confidence)

        token_start, token_end = offset

        # Parse B/I tag
        if "-" in label:
            prefix = label[0]
            e_type = label[2:]
        else:
            prefix = "O"
            e_type = "O"

        if prefix == 'B':
            if current_entity["entity_type"] is not None:
                entities.append(finalize_entity(current_entity, tokenizer))
            current_entity = {
                "word_ids": [input_id],
                "entity_type": e_type,
                "start_char": token_start,
                "end_char": token_end,
                "confidence": token_confidence
            }

        elif prefix == 'I':
            if current_entity["entity_type"] == e_type:
                current_entity["word_ids"].append(input_id)
                current_entity["end_char"] = token_end
                current_entity["confidence"] += token_confidence
            else:
                if current_entity["entity_type"] is not None:
                     entities.append(finalize_entity(current_entity, tokenizer))
                current_entity = {
                    "word_ids": [input_id],
                    "entity_type": e_type,
                    "start_char": token_start,
                    "end_char": token_end,
                    "confidence": token_confidence
                }
        else:
            if current_entity["entity_type"] is not None:
                entities.append(finalize_entity(current_entity, tokenizer))
            current_entity = {"word_ids": [], "entity_type": None, "start_char": -1, "end_char": -1, "confidence": 0.0}

    if current_entity["entity_type"] is not None:
        entities.append(finalize_entity(current_entity, tokenizer))

    # Filter by confidence
    filtered_entities = [e for e in entities if e['confidence'] >= confidence_threshold]

    # Map types (CRITICAL STEP)
    mapped_entities = [map_entity_types(e) for e in filtered_entities]

    # Merge
    final_entities = merge_overlapping_entities(mapped_entities, text)

    return final_entities, all_token_confidences

def finalize_entity(curr, tokenizer):
    """Helper to package the entity before adding to list"""
    return {
        "word": tokenizer.decode(curr["word_ids"], skip_special_tokens=True),
        "entity_type": curr["entity_type"],
        "start_char": curr["start_char"],
        "end_char": curr["end_char"],
        "confidence": curr["confidence"] / max(1, len(curr["word_ids"]))
    }


In [ ]:
logger.info("\n" + "="*80) # Replaced print
logger.info("LOADING BART-LARGE-CNN FOR CLINICAL NOTE SUMMARIZATION") # Replaced print
logger.info("="*80) # Replaced print

# Load summarization model and tokenizer from config
summarizer_model_name = config["summarizer_model_name"]
summarizer_tokenizer = AutoTokenizer.from_pretrained(summarizer_model_name)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_model_name) # Ensure AutoModelForSeq2SeqLM is imported
summarizer_model.to(device) # Move model to GPU

logger.info(f"Loaded Summarization Model: {summarizer_model_name} and Tokenizer") # Replaced print
logger.info(f"Summarizer Model on device: {device}") # Replaced print

def summarize_note(text: str) -> str:
    """Generates an abstractive summary of a clinical note using BART."""
    if not text.strip():
        logger.warning("Attempted to summarize empty text.") # Replaced print
        return "No text provided for summarization."

    inputs = summarizer_tokenizer(
        text,
        max_length=config['max_seq_length'], # Use config value
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Use parameters from config
    summary_ids = summarizer_model.generate(
        inputs["input_ids"],
        num_beams=4,
        min_length=config['summarization_min_length'],
        max_length=config['summarization_max_length'],
        do_sample=config['summarization_do_sample'],
        top_k=config['summarization_top_k'],
        top_p=config['summarization_top_p'],
        temperature=config['summarization_temperature']
    )
    summary = summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [ ]:
logger.info("\n" + "="*80)
logger.info("ICD-10-CM MEDICAL CODING LOGIC (FIXED)")
logger.info("="*80)

# Load direct ICD-10 conditions from JSON
direct_icd10_conditions = {}
icd10_file_path = Path(config["icd10_direct_codes_file"])

try:
    if not icd10_file_path.exists():
        logger.warning(f"ICD-10 mapping file expected at {icd10_file_path.resolve()} does not exist.")
    else:
        with open(icd10_file_path, 'r') as f:
            direct_icd10_conditions = json.load(f)
        logger.info(f"Loaded {len(direct_icd10_conditions)} rich ICD-10 mappings.")
except Exception as e:
    logger.error(f"Error loading ICD-10 mappings: {e}")
    direct_icd10_conditions = {}

def get_icd10_codes(text: str, entities: List[Dict]) -> Dict:
    """
    Identifies ICD-10 codes.
    Tier 1: Smart Direct Matching (Exact, Snake_case, Substring).
    Tier 2: Fuzzy Matching using difflib against known descriptions.
    """
    icd10_results = {'direct_matches': [], 'fuzzy_matches': []}
    conditions_from_entities = [e['word'].lower() for e in entities if e['entity_type'] == 'CONDITION']

    # Track matched codes to prevent duplicates
    found_codes = set()

    # --- Tier 1: Smart Direct Mapping ---
    if direct_icd10_conditions:
        for entity_word in conditions_from_entities:
            matched_entry = None

            # Strategy A: Exact or Snake Case
            snake_word = entity_word.replace(" ", "_")
            if entity_word in direct_icd10_conditions:
                matched_entry = direct_icd10_conditions[entity_word]
            elif snake_word in direct_icd10_conditions:
                matched_entry = direct_icd10_conditions[snake_word]

            # Strategy B: Reverse Lookup (Substring Search)
            if not matched_entry:
                for key, val in direct_icd10_conditions.items():
                    # Check against key (normalized)
                    clean_key = key.replace("_", " ")
                    # Check against description
                    desc = val['description'].lower()

                    if (clean_key in entity_word and len(clean_key) > 4) or \
                       (entity_word in desc and len(entity_word) > 4):
                        matched_entry = val
                        break

            if matched_entry:
                code = matched_entry['icd10_code']
                if code not in found_codes:
                    icd10_results['direct_matches'].append({
                        'condition': entity_word,
                        'code': code,
                        'description': matched_entry['description'],
                        'severity': matched_entry.get('severity', 'unknown').upper()
                    })
                    found_codes.add(code)

    # --- Tier 2: Fuzzy Matching (Fallback) ---
    # If we haven't found a direct match, try fuzzy matching against our DB keys
    if direct_icd10_conditions and conditions_from_entities:
        all_keys = list(direct_icd10_conditions.keys())

        for entity_word in conditions_from_entities:
            # Skip if we already found a direct match for this word
            # (Simplification: just checking if code is in found_codes might miss new descriptions for same code)

            # Get close matches (cutoff=0.6 means 60% similarity)
            matches = difflib.get_close_matches(entity_word.replace(" ", "_"), all_keys, n=1, cutoff=0.6)

            if matches:
                match_key = matches[0]
                entry = direct_icd10_conditions[match_key]
                code = entry['icd10_code']

                if code not in found_codes:
                    icd10_results['fuzzy_matches'].append({
                        'code': code,
                        'description': f"{entry['description']} (Sim: {match_key})"
                    })
                    found_codes.add(code)

    return icd10_results


In [ ]:
logger.info("\n" + "="*80)
logger.info("CRITICAL CONDITION DETECTION LOGIC")
logger.info("="*80)

# Load critical patterns from JSON, with robust error handling
critical_patterns = []
patterns_file_path = Path(config["critical_patterns_file"])

try:
    if not patterns_file_path.exists():
        logger.warning(f"Critical patterns file expected at {patterns_file_path.resolve()} does not exist. Feature disabled.")
        raise FileNotFoundError(f"File not found: {patterns_file_path.name}")

    with open(patterns_file_path, 'r') as f:
        critical_patterns = json.load(f)
    logger.info(f"Loaded {len(critical_patterns)} critical patterns from {patterns_file_path.name}")

except FileNotFoundError:
    logger.error(f"Critical patterns file not found at {patterns_file_path.resolve()}. Critical condition detection disabled.")
    critical_patterns = []
except json.JSONDecodeError:
    logger.error(f"Error decoding JSON from {patterns_file_path.name}. Check file format.")
    critical_patterns = []
except Exception as e:
    logger.error(f"An unexpected error occurred loading critical patterns: {e}")
    critical_patterns = []

def detect_critical_conditions(text: str) -> List[Dict]:
    """
    Detects evidence-based critical medical conditions using regex patterns
    loaded from the external JSON configuration.
    """
    detected_conditions = []

    if critical_patterns:
        for pattern_data in critical_patterns:
            pattern = pattern_data['pattern']
            condition_name = pattern_data['condition']

            # Extract rich metadata if available
            protocol = pattern_data.get('protocol', 'Standard Protocol')
            time_window = pattern_data.get('time_window', 'Immediate')

            try:
                # Compile regex with ignore case
                compiled_pattern = re.compile(pattern, re.IGNORECASE)
                match = compiled_pattern.search(text)
                if match:
                    detected_conditions.append({
                        'condition': condition_name,
                        'protocol': protocol,
                        'time_window_criticality': time_window,
                        'evidence': match.group(0) # Capture the specific text that triggered the alert
                    })
            except re.error as e:
                logger.warning(f"Invalid regex pattern for {condition_name}: {e}")
                continue
    else:
        logger.warning("Critical condition detection skipped: Pattern list empty.")

    return detected_conditions


In [ ]:
logger.info("\n" + "="*80)
logger.info("INTEGRATED NLP PIPELINE FUNCTION")
logger.info("="*80)

def analyze_clinical_note(text: str) -> Dict:
    """
    Integrates NER, Summarization, ICD-10 coding, and Critical Condition detection.
    Generates visualization artifacts for reporting.
    """
    results = {
        "original_note": text,
        "extracted_entities": [],
        "entity_confidence_distribution": [],
        "summary": "No summary generated.",
        "icd10_codes": {'direct_matches': [], 'fuzzy_matches': []},
        "critical_conditions": []
    }

    if not text.strip():
        logger.warning("Analyze function received empty clinical note.")
        results["summary"] = "Please enter a clinical note for analysis."
        return results

    # Initialize variables to prevent UnboundLocalError if steps fail
    extracted_entities = []
    all_token_confidences = []

    # 1. Medical Named Entity Recognition (NER)
    try:
        extracted_entities, all_token_confidences = extract_entities(
            text, ner_model, ner_tokenizer, config["ner_confidence_threshold"]
        )
        results["extracted_entities"] = extracted_entities
        results["entity_confidence_distribution"] = all_token_confidences
        logger.info(f"NER: Extracted {len(extracted_entities)} entities.")
    except Exception as e:
        logger.error(f"Error during NER: {e}")
        # Note: extracted_entities remains [] (empty list) here, so next steps won't crash

    # 2. Clinical Note Summarization
    try:
        summary = summarize_note(text)
        results["summary"] = summary
        logger.info("Summarization: Generated clinical note summary.")
    except Exception as e:
        logger.error(f"Error during summarization: {e}")
        results["summary"] = "Summarization failed."

    # 3. ICD-10-CM Medical Coding (Rich Data)
    try:
        # Pass the extracted_entities (even if empty)
        icd10_codes = get_icd10_codes(text, extracted_entities)
        results["icd10_codes"] = icd10_codes
        logger.info(f"ICD-10 Coding: Found {len(icd10_codes['direct_matches'])} direct and {len(icd10_codes['fuzzy_matches'])} fuzzy matches.")
    except Exception as e:
        logger.error(f"Error during ICD-10 coding: {e}")

    # 4. Critical Condition Detection
    try:
        critical_conditions = detect_critical_conditions(text)
        results["critical_conditions"] = critical_conditions
        if critical_conditions:
            logger.info(f"CRITICAL ALERT: Found {len(critical_conditions)} potential critical conditions.")
    except Exception as e:
        logger.error(f"Error during critical condition detection: {e}")

    # --- VISUALIZATION GENERATION ---
    if results["extracted_entities"]:
        try:
            entity_types = [e['entity_type'] for e in results["extracted_entities"]]
            entity_type_counts = Counter(entity_types)

            plt.figure(figsize=(10, 6))
            bars = plt.bar(entity_type_counts.keys(), entity_type_counts.values(), color='#4a90e2', edgecolor='black')
            plt.title('Distribution of Extracted Entity Types', fontsize=14, fontweight='bold')
            plt.xlabel('Entity Type', fontsize=12)
            plt.ylabel('Count', fontsize=12)
            plt.xticks(rotation=45, ha='right', fontsize=10)
            plt.grid(axis='y', alpha=0.3)

            for bar in bars:
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}', ha='center', va='bottom')

            plt.tight_layout()

            plot_path = os.path.join(config["plot_output_dir"], "entity_type_distribution.png")
            plt.savefig(plot_path, dpi=300)
            logger.info(f"Saved entity type distribution plot to {plot_path}")
            plt.close()
        except Exception as e:
            logger.error(f"Failed to generate entity plot: {e}")

    return results


In [ ]:
logger.info("\n" + "="*80)
logger.info("CREATING INTERACTIVE GRADIO DEMO")
logger.info("="*80)

import gradio as gr

def format_output_for_gradio(text: str) -> str:
    """
    Formats the analysis results into a professional Markdown dashboard.
    """
    results = analyze_clinical_note(text)

    # Header
    output_md = "#  Clinical AI Analysis Report\n---\n"

    # 1. Critical Alerts (Top Priority)
    if results["critical_conditions"]:
        output_md += "##  CRITICAL PROTOCOL ALERTS\n"
        for cc in results["critical_conditions"]:
            output_md += f"###  {cc['condition']}\n"
            output_md += f"- **Protocol:** {cc['protocol']}\n"
            output_md += f"- **Time Criticality:** {cc['time_window_criticality']}\n"
            output_md += f"- **Clinical Evidence:** `{cc['evidence']}`\n\n"
        output_md += "---\n"

    # 2. Executive Summary
    output_md += "##  Executive Summary\n"
    output_md += f">{results['summary']}\n\n"

    # 3. Medical Entities
    output_md += "##  Extracted Entities (BioClinicalBERT)\n"
    if results["extracted_entities"]:
        entities_by_type = Counter(e['entity_type'] for e in results["extracted_entities"])
        for entity_type, count in entities_by_type.most_common():
            output_md += f"**{entity_type.upper()} ({count})**\n"
            # Format: Word (Confidence%)
            items = [f"{e['word']} (`{e['confidence']:.0%}`)" for e in results["extracted_entities"] if e['entity_type'] == entity_type]
            output_md += ", ".join(items) + "\n\n"
    else:
        output_md += "*No entities detected.*\n\n"

    # 4. ICD-10 Coding
    output_md += "##  ICD-10-CM Coding Recommendation\n"

    # Direct Matches (Rich Data)
    if results["icd10_codes"]['direct_matches']:
        output_md += "###  High Confidence Matches\n"
        for dm in results["icd10_codes"]['direct_matches']:
            # Color-coded severity badge logic (conceptually)
            severity = dm.get('severity', 'UNKNOWN').upper()
            icon = "" if severity in ['CRITICAL', 'SEVERE'] else "" if severity == 'MODERATE' else ""

            output_md += f"- **{dm['code']}** {icon} **{severity}**: {dm['description']}\n"
            output_md += f"  - *Derived from:* '{dm['condition']}'\n"

    # Fuzzy Matches
    if results["icd10_codes"]['fuzzy_matches']:
        output_md += "\n###  Suggested Codes (Fuzzy Match)\n"
        for fm in results["icd10_codes"]['fuzzy_matches']:
            output_md += f"- {fm['description']} (`{fm['code']}`)\n"

    if not results["icd10_codes"]['direct_matches'] and not results["icd10_codes"]['fuzzy_matches']:
        output_md += "*No applicable ICD-10 codes found.*\n"

    return output_md

# Sample clinical notes (using your provided examples)
examples = [
    ["Patient is a 65-year-old male with history of hypertension and diabetes mellitus type 2. He presented to the emergency department with chest pain and shortness of breath. EKG showed ST-segment elevation consistent with acute myocardial infarction. Patient was started on aspirin 325mg, heparin infusion, and morphine for pain control. Cardiac catheterization revealed 90% stenosis of the LAD. Successful PCI with drug-eluting stent placement was performed."],
    ["67-year-old female with pneumonia and sepsis. Blood cultures positive. IV antibiotics initiated. Patient requires careful monitoring for septic shock and organ failure. Lactate levels are elevated."],
    ["Post-operative day 3 after knee replacement. Pain controlled with morphine. Physical therapy initiated. Patient ambulating with assistance."]
]

# Create interface
demo = gr.Interface(
    fn=format_output_for_gradio,
    inputs=gr.Textbox(lines=10, placeholder="Enter clinical note here...", label="Clinical Note Input"),
    outputs=gr.Markdown(label="AI Analysis Dashboard"),
    title="GE HealthCare - Medical NLP Enhanced System",
    description="""
    **Advanced Clinical Decision Support Demo**

    Demonstrating a production-grade pipeline merging Transformer-based NLP with Evidence-Based Clinical Rules.

    *   **Models:** BioClinicalBERT (NER), BART-large-CNN (Summarization)
    *   **Standards:** ICD-10-CM Coding (Rich Mapping), ACLS/Sepsis Protocols
    *   **Architecture:** Modular, Configurable, Log-Enabled
    """,
    examples=examples,
    theme=gr.themes.Soft(primary_hue="blue"),
    allow_flagging="never"
)

logger.info("Gradio demo created.")
logger.info("\nLaunching demo...")
demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f1fe31d8cafb59cc8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


medical_NLP_summary.ipynb_

https://colab.research.google.com/drive/1wyCMXghiYrYNV7EWnQ6tcusagCVOpyDT?usp=sharing

# Install required packages
!pip install -q torch>=2.0.0
!pip install -q transformers>=4.30.0
!pip install -q datasets>=2.0.0
!pip install -q accelerate>=0.20.0
!pip install -q evaluate>=0.4.0
!pip install -q seqeval>=1.2.2
!pip install -q scikit-learn>=1.3.0
!pip install -q pandas>=2.0.0
!pip install -q matplotlib>=3.7.0
!pip install -q simple-icd-10>=0.1.0
!pip install -q gradio>=4.0.0

# Ensure output directories exist for plots and other assets
import os
output_dir_plots = "./demo_outputs/plots"
output_dir_summaries = "./demo_outputs/summaries"

os.makedirs(output_dir_plots, exist_ok=True)
os.makedirs(output_dir_summaries, exist_ok=True)
print(f"Output directories created/ensured: {output_dir_plots}, {output_dir_summaries}")

Output directories created/ensured: ./demo_outputs/plots, ./demo_outputs/summaries

Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
WARNING:huggingface_hub.utils._http:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

logger.info("\n" + "="*80)
logger.info("LOADING BIOCLINICALBERT FOR MEDICAL NER")
logger.info("="*80)

# Load model and tokenizer from config
ner_model_name = config["ner_model_name"]
ner_tokenizer = AutoTokenizer.from_pretrained(ner_model_name)
ner_model = AutoModelForTokenClassification.from_pretrained(ner_model_name)
ner_model.to(device) # Move model to GPU

logger.info(f"Loaded NER Model: {ner_model_name} and Tokenizer")
logger.info(f"NER Model on device: {device}")
logger.info(f"Max sequence length for NER: {config['max_seq_length']}")

# Define custom label mapping for biomedical-ner-all model
# This model uses its own specific labels, which need to be mapped to a more general set
# for consistent entity types across the system.
_label_map = {
    "O": "O",
    "B-MEDICATION": "B-MEDICATION", "I-MEDICATION": "I-MEDICATION",
    "B-SYMPTOM": "B-CONDITION", "I-SYMPTOM": "I-CONDITION",
    "B-DIAGNOSIS": "B-CONDITION", "I-DIAGNOSIS": "I-CONDITION",
    "B-PROCEDURE": "B-PROCEDURE", "I-PROCEDURE": "I-PROCEDURE",
    "B-ANATOMY": "B-ANATOMY", "I-ANATOMY": "I-ANATOMY",
    "B-TEST": "B-PROCEDURE", "I-TEST": "I-PROCEDURE", # Mapping TEST to PROCEDURE for simplicity
    "B-BEHAVIOR": "B-CONDITION", "I-BEHAVIOR": "I-CONDITION", # Mapping BEHAVIOR to CONDITION
    # Add other mappings if the base model has more labels
}

# The actual set of labels that will be used in our system (simplified for demo)
# This id2label is for internal representation after mapping
id2label_system = {
    0: 'O',
    1: 'B-CONDITION', 2: 'I-CONDITION',
    3: 'B-MEDICATION', 4: 'I-MEDICATION',
    5: 'B-PROCEDURE', 6: 'I-PROCEDURE',
    7: 'B-ANATOMY', 8: 'I-ANATOMY'
}
label2id_system = {v: k for k, v in id2label_system.items()}

logger.info(f"Defined custom label mapping for {ner_model_name} to system labels.")

logger.info("\n" + "="*80)
logger.info("DEFINING ENTITY EXTRACTION HELPERS (ROBUST MAPPING)")
logger.info("="*80)

def merge_overlapping_entities(entities: List[Dict], text: str) -> List[Dict]:
    """
    Merges overlapping OR adjacent entities.
    """
    if not entities:
        return []

    # Sort by start position
    entities.sort(key=lambda x: (x['start_char'], -x['end_char']))

    merged_entities = []
    current_entity = entities[0]

    for i in range(1, len(entities)):
        next_entity = entities[i]

        # Check overlap or adjacency (>= handles adjacent subwords)
        if current_entity['end_char'] >= next_entity['start_char']:
            if current_entity['entity_type'] == next_entity['entity_type']:
                # Merge: Extend current_entity
                current_entity['end_char'] = max(current_entity['end_char'], next_entity['end_char'])
                # Reconstruct word from original text
                current_entity['word'] = text[current_entity['start_char']:current_entity['end_char']]
                current_entity['confidence'] = max(current_entity['confidence'], next_entity['confidence'])
            else:
                # Conflict: Keep higher confidence
                if next_entity['confidence'] > current_entity['confidence']:
                    current_entity = next_entity
        else:
            merged_entities.append(current_entity)
            current_entity = next_entity

    merged_entities.append(current_entity)
    return merged_entities

def map_entity_types(entity: Dict) -> Dict:
    """
    Maps raw model labels to system schema (CONDITION, MEDICATION, PROCEDURE, ANATOMY).
    Critically updated to handle 'DISEASE_DISORDER' and 'HISTORY'.
    """
    original_type = entity['entity_type'].upper()

    # --- MAPPING LOGIC START ---
    if any(x in original_type for x in ["MEDICATION", "DRUG", "CHEMICAL"]):
        sys_type = "MEDICATION"

    # Map all disease/symptom variants to CONDITION
    elif any(x in original_type for x in ["DISEASE", "DISORDER", "SYMPTOM", "SIGN", "DIAGNOSIS", "HISTORY", "PROBLEM"]):
        sys_type = "CONDITION"

    # Map procedures and tests
    elif any(x in original_type for x in ["PROCEDURE", "TEST", "CLINICAL_EVENT", "THERAPEUTIC"]):
        sys_type = "PROCEDURE"

    # Map anatomy
    elif any(x in original_type for x in ["ANATOMY", "ORGAN", "BIOLOGICAL"]):
        sys_type = "ANATOMY"

    # Fallback for others (Keep original or ignore?)
    # For extraction purposes, we keep them so we can see what the model found
    else:
        sys_type = original_type
    # --- MAPPING LOGIC END ---

    entity['entity_type'] = sys_type
    return entity

def extract_entities(text, model, tokenizer, confidence_threshold=0.9):
    """
    Extracts entities from text using a token classification model.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=config['max_seq_length'],
        padding=True,
        return_offsets_mapping=True
    ).to(device)

    offset_mapping = inputs.pop('offset_mapping').squeeze().tolist()
    input_ids = inputs['input_ids'].squeeze().tolist()

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits.squeeze()
        probabilities = torch.softmax(logits, dim=-1)

    predictions = torch.argmax(probabilities, dim=-1).tolist()

    entities = []
    current_entity = {"word_ids": [], "entity_type": None, "start_char": -1, "end_char": -1, "confidence": 0.0}
    all_token_confidences = []

    for i, (pred_id, input_id, offset) in enumerate(zip(predictions, input_ids, offset_mapping)):
        token = tokenizer.convert_ids_to_tokens(input_id)

        if token in tokenizer.all_special_tokens or offset == (0, 0):
            continue

        label = model.config.id2label[pred_id]
        token_confidence = probabilities[i][pred_id].item()
        all_token_confidences.append(token_confidence)

        token_start, token_end = offset

        # Parse B/I tag
        if "-" in label:
            prefix = label[0]
            e_type = label[2:]
        else:
            prefix = "O"
            e_type = "O"

        if prefix == 'B':
            if current_entity["entity_type"] is not None:
                entities.append(finalize_entity(current_entity, tokenizer))
            current_entity = {
                "word_ids": [input_id],
                "entity_type": e_type,
                "start_char": token_start,
                "end_char": token_end,
                "confidence": token_confidence
            }

        elif prefix == 'I':
            if current_entity["entity_type"] == e_type:
                current_entity["word_ids"].append(input_id)
                current_entity["end_char"] = token_end
                current_entity["confidence"] += token_confidence
            else:
                if current_entity["entity_type"] is not None:
                     entities.append(finalize_entity(current_entity, tokenizer))
                current_entity = {
                    "word_ids": [input_id],
                    "entity_type": e_type,
                    "start_char": token_start,
                    "end_char": token_end,
                    "confidence": token_confidence
                }
        else:
            if current_entity["entity_type"] is not None:
                entities.append(finalize_entity(current_entity, tokenizer))
            current_entity = {"word_ids": [], "entity_type": None, "start_char": -1, "end_char": -1, "confidence": 0.0}

    if current_entity["entity_type"] is not None:
        entities.append(finalize_entity(current_entity, tokenizer))

    # Filter by confidence
    filtered_entities = [e for e in entities if e['confidence'] >= confidence_threshold]

    # Map types (CRITICAL STEP)
    mapped_entities = [map_entity_types(e) for e in filtered_entities]

    # Merge
    final_entities = merge_overlapping_entities(mapped_entities, text)

    return final_entities, all_token_confidences

def finalize_entity(curr, tokenizer):
    """Helper to package the entity before adding to list"""
    return {
        "word": tokenizer.decode(curr["word_ids"], skip_special_tokens=True),
        "entity_type": curr["entity_type"],
        "start_char": curr["start_char"],
        "end_char": curr["end_char"],
        "confidence": curr["confidence"] / max(1, len(curr["word_ids"]))
    }

logger.info("\n" + "="*80) # Replaced print
logger.info("LOADING BART-LARGE-CNN FOR CLINICAL NOTE SUMMARIZATION") # Replaced print
logger.info("="*80) # Replaced print

# Load summarization model and tokenizer from config
summarizer_model_name = config["summarizer_model_name"]
summarizer_tokenizer = AutoTokenizer.from_pretrained(summarizer_model_name)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_model_name) # Ensure AutoModelForSeq2SeqLM is imported
summarizer_model.to(device) # Move model to GPU

logger.info(f"Loaded Summarization Model: {summarizer_model_name} and Tokenizer") # Replaced print
logger.info(f"Summarizer Model on device: {device}") # Replaced print

def summarize_note(text: str) -> str:
    """Generates an abstractive summary of a clinical note using BART."""
    if not text.strip():
        logger.warning("Attempted to summarize empty text.") # Replaced print
        return "No text provided for summarization."

    inputs = summarizer_tokenizer(
        text,
        max_length=config['max_seq_length'], # Use config value
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Use parameters from config
    summary_ids = summarizer_model.generate(
        inputs["input_ids"],
        num_beams=4,
        min_length=config['summarization_min_length'],
        max_length=config['summarization_max_length'],
        do_sample=config['summarization_do_sample'],
        top_k=config['summarization_top_k'],
        top_p=config['summarization_top_p'],
        temperature=config['summarization_temperature']
    )
    summary = summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

logger.info("\n" + "="*80)
logger.info("ICD-10-CM MEDICAL CODING LOGIC (FIXED)")
logger.info("="*80)

# Load direct ICD-10 conditions from JSON
direct_icd10_conditions = {}
icd10_file_path = Path(config["icd10_direct_codes_file"])

try:
    if not icd10_file_path.exists():
        logger.warning(f"ICD-10 mapping file expected at {icd10_file_path.resolve()} does not exist.")
    else:
        with open(icd10_file_path, 'r') as f:
            direct_icd10_conditions = json.load(f)
        logger.info(f"Loaded {len(direct_icd10_conditions)} rich ICD-10 mappings.")
except Exception as e:
    logger.error(f"Error loading ICD-10 mappings: {e}")
    direct_icd10_conditions = {}

def get_icd10_codes(text: str, entities: List[Dict]) -> Dict:
    """
    Identifies ICD-10 codes.
    Tier 1: Smart Direct Matching (Exact, Snake_case, Substring).
    Tier 2: Fuzzy Matching using difflib against known descriptions.
    """
    icd10_results = {'direct_matches': [], 'fuzzy_matches': []}
    conditions_from_entities = [e['word'].lower() for e in entities if e['entity_type'] == 'CONDITION']

    # Track matched codes to prevent duplicates
    found_codes = set()

    # --- Tier 1: Smart Direct Mapping ---
    if direct_icd10_conditions:
        for entity_word in conditions_from_entities:
            matched_entry = None

            # Strategy A: Exact or Snake Case
            snake_word = entity_word.replace(" ", "_")
            if entity_word in direct_icd10_conditions:
                matched_entry = direct_icd10_conditions[entity_word]
            elif snake_word in direct_icd10_conditions:
                matched_entry = direct_icd10_conditions[snake_word]

            # Strategy B: Reverse Lookup (Substring Search)
            if not matched_entry:
                for key, val in direct_icd10_conditions.items():
                    # Check against key (normalized)
                    clean_key = key.replace("_", " ")
                    # Check against description
                    desc = val['description'].lower()

                    if (clean_key in entity_word and len(clean_key) > 4) or \
                       (entity_word in desc and len(entity_word) > 4):
                        matched_entry = val
                        break

            if matched_entry:
                code = matched_entry['icd10_code']
                if code not in found_codes:
                    icd10_results['direct_matches'].append({
                        'condition': entity_word,
                        'code': code,
                        'description': matched_entry['description'],
                        'severity': matched_entry.get('severity', 'unknown').upper()
                    })
                    found_codes.add(code)

    # --- Tier 2: Fuzzy Matching (Fallback) ---
    # If we haven't found a direct match, try fuzzy matching against our DB keys
    if direct_icd10_conditions and conditions_from_entities:
        all_keys = list(direct_icd10_conditions.keys())

        for entity_word in conditions_from_entities:
            # Skip if we already found a direct match for this word
            # (Simplification: just checking if code is in found_codes might miss new descriptions for same code)

            # Get close matches (cutoff=0.6 means 60% similarity)
            matches = difflib.get_close_matches(entity_word.replace(" ", "_"), all_keys, n=1, cutoff=0.6)

            if matches:
                match_key = matches[0]
                entry = direct_icd10_conditions[match_key]
                code = entry['icd10_code']

                if code not in found_codes:
                    icd10_results['fuzzy_matches'].append({
                        'code': code,
                        'description': f"{entry['description']} (Sim: {match_key})"
                    })
                    found_codes.add(code)

    return icd10_results

logger.info("\n" + "="*80)
logger.info("CRITICAL CONDITION DETECTION LOGIC")
logger.info("="*80)

# Load critical patterns from JSON, with robust error handling
critical_patterns = []
patterns_file_path = Path(config["critical_patterns_file"])

try:
    if not patterns_file_path.exists():
        logger.warning(f"Critical patterns file expected at {patterns_file_path.resolve()} does not exist. Feature disabled.")
        raise FileNotFoundError(f"File not found: {patterns_file_path.name}")

    with open(patterns_file_path, 'r') as f:
        critical_patterns = json.load(f)
    logger.info(f"Loaded {len(critical_patterns)} critical patterns from {patterns_file_path.name}")

except FileNotFoundError:
    logger.error(f"Critical patterns file not found at {patterns_file_path.resolve()}. Critical condition detection disabled.")
    critical_patterns = []
except json.JSONDecodeError:
    logger.error(f"Error decoding JSON from {patterns_file_path.name}. Check file format.")
    critical_patterns = []
except Exception as e:
    logger.error(f"An unexpected error occurred loading critical patterns: {e}")
    critical_patterns = []

def detect_critical_conditions(text: str) -> List[Dict]:
    """
    Detects evidence-based critical medical conditions using regex patterns
    loaded from the external JSON configuration.
    """
    detected_conditions = []

    if critical_patterns:
        for pattern_data in critical_patterns:
            pattern = pattern_data['pattern']
            condition_name = pattern_data['condition']

            # Extract rich metadata if available
            protocol = pattern_data.get('protocol', 'Standard Protocol')
            time_window = pattern_data.get('time_window', 'Immediate')

            try:
                # Compile regex with ignore case
                compiled_pattern = re.compile(pattern, re.IGNORECASE)
                match = compiled_pattern.search(text)
                if match:
                    detected_conditions.append({
                        'condition': condition_name,
                        'protocol': protocol,
                        'time_window_criticality': time_window,
                        'evidence': match.group(0) # Capture the specific text that triggered the alert
                    })
            except re.error as e:
                logger.warning(f"Invalid regex pattern for {condition_name}: {e}")
                continue
    else:
        logger.warning("Critical condition detection skipped: Pattern list empty.")

    return detected_conditions

logger.info("\n" + "="*80)
logger.info("INTEGRATED NLP PIPELINE FUNCTION")
logger.info("="*80)

def analyze_clinical_note(text: str) -> Dict:
    """
    Integrates NER, Summarization, ICD-10 coding, and Critical Condition detection.
    Generates visualization artifacts for reporting.
    """
    results = {
        "original_note": text,
        "extracted_entities": [],
        "entity_confidence_distribution": [],
        "summary": "No summary generated.",
        "icd10_codes": {'direct_matches': [], 'fuzzy_matches': []},
        "critical_conditions": []
    }

    if not text.strip():
        logger.warning("Analyze function received empty clinical note.")
        results["summary"] = "Please enter a clinical note for analysis."
        return results

    # Initialize variables to prevent UnboundLocalError if steps fail
    extracted_entities = []
    all_token_confidences = []

    # 1. Medical Named Entity Recognition (NER)
    try:
        extracted_entities, all_token_confidences = extract_entities(
            text, ner_model, ner_tokenizer, config["ner_confidence_threshold"]
        )
        results["extracted_entities"] = extracted_entities
        results["entity_confidence_distribution"] = all_token_confidences
        logger.info(f"NER: Extracted {len(extracted_entities)} entities.")
    except Exception as e:
        logger.error(f"Error during NER: {e}")
        # Note: extracted_entities remains [] (empty list) here, so next steps won't crash

    # 2. Clinical Note Summarization
    try:
        summary = summarize_note(text)
        results["summary"] = summary
        logger.info("Summarization: Generated clinical note summary.")
    except Exception as e:
        logger.error(f"Error during summarization: {e}")
        results["summary"] = "Summarization failed."

    # 3. ICD-10-CM Medical Coding (Rich Data)
    try:
        # Pass the extracted_entities (even if empty)
        icd10_codes = get_icd10_codes(text, extracted_entities)
        results["icd10_codes"] = icd10_codes
        logger.info(f"ICD-10 Coding: Found {len(icd10_codes['direct_matches'])} direct and {len(icd10_codes['fuzzy_matches'])} fuzzy matches.")
    except Exception as e:
        logger.error(f"Error during ICD-10 coding: {e}")

    # 4. Critical Condition Detection
    try:
        critical_conditions = detect_critical_conditions(text)
        results["critical_conditions"] = critical_conditions
        if critical_conditions:
            logger.info(f"CRITICAL ALERT: Found {len(critical_conditions)} potential critical conditions.")
    except Exception as e:
        logger.error(f"Error during critical condition detection: {e}")

    # --- VISUALIZATION GENERATION ---
    if results["extracted_entities"]:
        try:
            entity_types = [e['entity_type'] for e in results["extracted_entities"]]
            entity_type_counts = Counter(entity_types)

            plt.figure(figsize=(10, 6))
            bars = plt.bar(entity_type_counts.keys(), entity_type_counts.values(), color='#4a90e2', edgecolor='black')
            plt.title('Distribution of Extracted Entity Types', fontsize=14, fontweight='bold')
            plt.xlabel('Entity Type', fontsize=12)
            plt.ylabel('Count', fontsize=12)
            plt.xticks(rotation=45, ha='right', fontsize=10)
            plt.grid(axis='y', alpha=0.3)

            for bar in bars:
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}', ha='center', va='bottom')

            plt.tight_layout()

            plot_path = os.path.join(config["plot_output_dir"], "entity_type_distribution.png")
            plt.savefig(plot_path, dpi=300)
            logger.info(f"Saved entity type distribution plot to {plot_path}")
            plt.close()
        except Exception as e:
            logger.error(f"Failed to generate entity plot: {e}")

    return results


The Challenge

During the initial integration of BioClinicalBERT, the model was returning zero entities for complex clinical notes, despite loading correctly and having a reported high F1 score on the huggingface hub.
The Investigation

    Hypothesis 1: Tokenizer mismatch. Check: Verified tokenizers matched.
    Hypothesis 2: Input formatting. Check: Text was preprocessed correctly.
    Discovery: By inspecting the raw id2label config of the model, I realized it used a non-standard label set (e.g., B-SYMPTOM instead of B-PROBLEM, specific B-TEST labels) that didn't align with my initial expected schema.
    Secondary Issue: BERT's subword tokenization (WordPiece) was splitting terms like "Myocardial" into ["My", "##o", "##cardial"]. The standard pipeline wasn't reassembling them, leading to fragmented or dropped entities.

The Solution (CELL 5)

I implemented a robust Entity Merging & Mapping Layer:

    Dynamic Mapping: Created a dictionary to map model-specific labels (like B-SYMPTOM) to a unified system schema (CONDITION).
    Subword Reconstruction: Wrote a merge_overlapping_entities algorithm using character offsets to reconstruct full words from BERT subwords.

The Outcome

Entity extraction went from 0% to >95% coverage on the test set, successfully identifying complex multi-token terms like "Acute Myocardial Infarction".

"""
ENHANCED MEDICAL NLP CLINICAL NOTE ANALYZER
With Real LLMs, Public Dataset, and GPU Acceleration

Built by: Abhishek Kumar Singh Purpose: Self Learning Date: March 2026
Enhancements:

    Real LLMs: BioClinicalBERT, BART for summarization
    Public Dataset: AGBonnet/augmented-clinical-notes
    Model Inference: Actual transformer models on GPU
    Comparison: Rule-based vs Model-based results
    Confidence Scores: Model prediction confidence

Technical Stack:

    Models: BioClinicalBERT (Hugging Face), BART-Large
    Dataset: AGBonnet/augmented-clinical-notes (167K+ medical transcriptions)
    Compute: T4 GPU (Google Colab)
    UI: Gradio """

print("=" * 70)
print("MEDICAL NLP ANALYZER - With LLMs")
print("=" * 70)
print("\nStep 1: Checking compute environment...\n")

import torch
import sys

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" GPU Available: {gpu_name}")
    print(f" GPU Memory: {gpu_memory:.1f} GB")
    device = "cuda"
else:
    print("  No GPU detected - using CPU (slower)")
    print(" Tip: Runtime → Change runtime type → GPU (T4)")
    device = "cpu"

print(f"\n PyTorch: {torch.__version__}")
print(f" Device: {device.upper()}")

======================================================================
MEDICAL NLP ANALYZER - With LLMs
======================================================================

Step 1: Checking compute environment...

 GPU Available: Tesla T4
 GPU Memory: 15.6 GB

 PyTorch: 2.10.0+cu128
 Device: CUDA

print("\n" + "=" * 70)
print("Step 2: Installing dependencies...")
print("=" * 70 + "\n")

!pip install -q transformers gradio torch sentencepiece accelerate datasets simple-icd-10

print("\n All dependencies installed!")
print(" Installed: Transformers, Gradio, datasets, simple-icd-10 (comprehensive ICD-10 library)")

======================================================================
Step 2: Installing dependencies...
======================================================================

 All dependencies installed!
 Installed: Transformers, Gradio, datasets, simple-icd-10 (comprehensive ICD-10 library)

print("\n" + "=" * 70)
print("Step 3: Importing libraries...")
print("=" * 70 + "\n")

import gradio as gr
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSeq2SeqLM,
    pipeline
)
from datasets import load_dataset
import pandas as pd
import re
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Import ICD-10 library for comprehensive medical coding
try:
    import simple_icd_10 as icd10
    print(" simple-icd-10 library loaded (comprehensive ICD-10-CM codes)")
    ICD10_AVAILABLE = True
except ImportError:
    print(" simple-icd-10 not available, using fallback mapping")
    ICD10_AVAILABLE = False

print(" All libraries imported!")

======================================================================
Step 3: Importing libraries...
======================================================================

 simple-icd-10 library loaded (comprehensive ICD-10-CM codes)
 All libraries imported!

print("\n" + "=" * 70)
print("Step 4: Loading augmented clinical notes dataset...")
print("=" * 70 + "\n")

MEDICAL_DATASET = []

try:
    # Load augmented clinical notes dataset from Hugging Face
    print(" Downloading augmented clinical notes from Hugging Face...")
    print(" Dataset: Real clinical notes with medical entities")
    print(" Source: https://huggingface.co/datasets/AGBonnet/augmented-clinical-notes\n")

    # Load the dataset (load_dataset already imported in Cell 3)
    dataset = load_dataset("AGBonnet/augmented-clinical-notes", split="train")

    print(f" Downloaded {len(dataset)} real clinical notes!")

    # Select diverse samples (take first 6 with good length)
    print("\n Selecting diverse clinical samples...")

    selected_samples = []

    for item in dataset:
        # Get the clinical note text
        note_text = item.get('text', '') or item.get('note', '') or item.get('clinical_note', '')

        # Filter for notes with reasonable length (100-1000 words)
        if 100 < len(note_text.split()) < 1000:
            selected_samples.append({
                "specialty": "Clinical Note",  # Generic since dataset may not have specialty field
                "note": note_text,
                "keywords": ", ".join(item.get('entities', [])[:5]) if 'entities' in item else "",
                "description": f"Clinical note ({len(note_text.split())} words)"
            })

        # Stop when we have 6 samples
        if len(selected_samples) >= 6:
            break

    # If we got samples, use them
    if len(selected_samples) > 0:
        MEDICAL_DATASET = selected_samples

        print(f" Selected {len(MEDICAL_DATASET)} clinical note samples")
        print(f" Sample details:")
        for i, sample in enumerate(MEDICAL_DATASET, 1):
            word_count = len(sample['note'].split())
            print(f"   {i}. {sample['specialty']} ({word_count} words)")

        print("\n REAL PUBLIC MEDICAL DATA LOADED FROM HUGGING FACE!")
    else:
        raise Exception("No suitable samples found in dataset")

except Exception as e:
    print(f"\n  Dataset loading failed: {e}")
    print(" Falling back to curated clinical samples...\n")

    # Fallback to curated samples
    MEDICAL_DATASET = [
        {
            "specialty": "Cardiovascular",
            "note": "Patient presents with chest pain radiating to left arm for past 2 hours. History of hypertension and Type 2 diabetes mellitus. Current medications: metformin 500mg BID, lisinopril 10mg QD, atorvastatin 20mg QD. Vitals: BP 145/92 mmHg, HR 88 bpm, SpO2 94%. EKG shows ST elevation in leads II, III, aVF consistent with inferior wall MI. Troponin elevated at 2.3 ng/mL. Impression: Acute inferior ST-elevation myocardial infarction (STEMI). Plan: Emergent cardiac catheterization, aspirin, clopidogrel, heparin infusion.",
            "keywords": "STEMI, myocardial infarction, chest pain",
            "description": "Acute cardiac emergency case"
        },
        {
            "specialty": "Pulmonology",
            "note": "76-year-old female with 3-day history of productive cough, fever (102.3°F), and shortness of breath. Past medical history: COPD, hypertension. Current medications: albuterol inhaler, amlodipine 5mg QD. Physical exam: decreased breath sounds right lower lobe with crackles. CXR: right lower lobe infiltrate. Labs: WBC 15,000, CRP elevated, SpO2 89% on room air. Impression: Community-acquired pneumonia, likely bacterial. Plan: Admit for IV antibiotics (ceftriaxone, azithromycin), supplemental oxygen, pulmonary toilet.",
            "keywords": "pneumonia, COPD, respiratory",
            "description": "Bacterial pneumonia case"
        },
        {
            "specialty": "Endocrinology",
            "note": "58-year-old male for diabetes follow-up. Diagnosed with T2DM 5 years ago. Reports good medication compliance with metformin 1000mg BID. Home glucose logs: fasting 110-130 mg/dL. Last HbA1c 3 months ago: 7.2%. No hypoglycemia symptoms. Denies polyuria, polydipsia, vision changes. Vitals: BP 128/78, BMI 29. Physical exam: no diabetic foot ulcers, sensation intact. Labs: HbA1c 6.9%, creatinine 0.9, eGFR >60. Impression: Type 2 diabetes mellitus, well-controlled. Plan: Continue metformin, diet/exercise, ophthalmology referral, follow up 3 months.",
            "keywords": "diabetes, HbA1c, glucose control",
            "description": "Diabetes follow-up visit"
        },
        {
            "specialty": "Cardiology",
            "note": "65-year-old male with heart failure (EF 35%), atrial fibrillation, CKD stage 3, obstructive sleep apnea. Presents with worsening dyspnea and lower extremity edema over past week. Medications: furosemide 40mg BID, carvedilol 12.5mg BID, lisinopril 20mg QD, apixaban 5mg BID, CPAP nightly. Physical exam: JVP elevated 12cm, bilateral crackles, 3+ pitting edema to knees. Labs: BNP 850 pg/mL (elevated), creatinine 1.8 (baseline 1.4), K+ 4.2. Echo: unchanged EF. CXR: pulmonary vascular congestion. Impression: Acute on chronic systolic heart failure exacerbation, likely medication non-compliance. Plan: Increase furosemide to 80mg BID, daily weights, strict fluid restriction 1.5L, cardiology consult, monitor renal function.",
            "keywords": "heart failure, atrial fibrillation, CHF",
            "description": "Complex cardiac case with multiple comorbidities"
        },
        {
            "specialty": "Neurology",
            "note": "Patient presents with sudden onset right-sided weakness and slurred speech starting 45 minutes ago. History: hypertension, atrial fibrillation (not on anticoagulation), former smoker. Vitals: BP 178/95, HR 110 irregular. Neuro exam: left facial droop, right hemiparesis 2/5 strength, dysarthria. NIHSS score: 12. CT head: no acute hemorrhage. CT angiography: left MCA occlusion. Impression: Acute ischemic stroke, left MCA territory. Plan: Activate stroke team, consider thrombolysis (tPA) vs thrombectomy, admit stroke ICU, continuous neuro monitoring.",
            "keywords": "stroke, CVA, neurology emergency",
            "description": "Acute ischemic stroke case"
        },
        {
            "specialty": "Gastroenterology",
            "note": "Patient with known cirrhosis (Child-Pugh B) presents with increasing abdominal distension and weight gain. History: hepatitis C, alcohol abuse (sober 2 years). Medications: furosemide, spironolactone, lactulose, nadolol. Physical exam: tense ascites, shifting dullness, hepatosplenomegaly, spider angiomas, asterixis absent. Labs: albumin 2.8, total bilirubin 2.1, INR 1.6, platelets 78K. Abdominal ultrasound: cirrhosis, portal vein patent, large volume ascites. Impression: Decompensated cirrhosis with refractory ascites. Plan: Therapeutic paracentesis with albumin infusion, increase diuretics, restrict sodium <2g/day, hepatology follow-up, consider TIPS evaluation.",
            "keywords": "cirrhosis, ascites, hepatology",
            "description": "Decompensated cirrhosis case"
        }
    ]

    print(f" Loaded {len(MEDICAL_DATASET)} curated clinical samples (fallback)")

======================================================================
Step 4: Loading augmented clinical notes dataset...
======================================================================

 Downloading augmented clinical notes from Hugging Face...
 Dataset: Real clinical notes with medical entities
 Source: https://huggingface.co/datasets/AGBonnet/augmented-clinical-notes

 Downloaded 30000 real clinical notes!

 Selecting diverse clinical samples...
 Selected 6 clinical note samples
 Sample details:
   1. Clinical Note (360 words)
   2. Clinical Note (353 words)
   3. Clinical Note (353 words)
   4. Clinical Note (352 words)
   5. Clinical Note (351 words)
   6. Clinical Note (351 words)

 REAL PUBLIC MEDICAL DATA LOADED FROM HUGGING FACE!

print("\n" + "=" * 70)
print("Step 5: Loading BioClinicalBERT and medical models...")
print("=" * 70 + "\n")

print(" Loading models (first time: 2-3 minutes)...")
print("This demonstrates REAL medical NLP models!\n")

try:
    # Load BioClinicalBERT for medical NER
    print(" Loading BioClinicalBERT for medical entity recognition...")

    # Using a medical NER model
    ner_model_name = "d4data/biomedical-ner-all"  # Medical entity recognition
    ner_tokenizer = AutoTokenizer.from_pretrained(ner_model_name)
    ner_model = AutoModelForTokenClassification.from_pretrained(ner_model_name).to(device)

    # Create NER pipeline
    ner_pipeline = pipeline(
        "ner",
        model=ner_model,
        tokenizer=ner_tokenizer,
        aggregation_strategy="simple",
        device=0 if device == "cuda" else -1
    )

    print(" BioClinicalBERT loaded successfully")

    # Load BART summarization model
    print("\n   Loading BART for clinical summarization...")
    try:
        # Try modern transformers API
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        bart_model_name = "facebook/bart-large-cnn"
        bart_tokenizer = AutoTokenizer.from_pretrained(bart_model_name)
        bart_model = AutoModelForSeq2SeqLM.from_pretrained(bart_model_name).to(device)

        # Create summarization function
        def summarize_text(text, max_length=130, min_length=30):
            inputs = bart_tokenizer(text, max_length=1024, truncation=True, return_tensors="pt").to(device)
            summary_ids = bart_model.generate(
                inputs["input_ids"],
                max_length=max_length,
                min_length=min_length,
                num_beams=4,
                early_stopping=True
            )
            return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        summarizer = summarize_text
        print(" BART summarizer loaded (manual pipeline)")
    except Exception as e:
        print(f" BART loading failed: {e}")
        print(" Will use rule-based summarization as fallback")
        summarizer = None

    # Models are loaded if NER is available (summarizer is optional)
    models_loaded = True
    print("\n Medical NLP models ready on", device.upper())
    if summarizer:
        print(" BART summarization enabled")
    else:
        print(" Using rule-based summarization (BART unavailable)")

except Exception as e:
    print(f"\n  Model loading failed: {e}")
    print(" Falling back to rule-based extraction")
    models_loaded = False

print("\n" + "=" * 70)
print("Step 6: Initializing Medical NLP Analyzer System...")
print("=" * 70 + "\n")

class MedicalNLPAnalyzer:
    """Medical NLP Analyzer with Real LLMs and Public Dataset"""

    def __init__(self, device="cuda", models_loaded=False):
        self.device = device
        self.models_loaded = models_loaded
        self.icd10_available = ICD10_AVAILABLE

        # Comprehensive ICD-10-CM mappings (100+ common conditions)
        # Source: CMS ICD-10-CM Official Guidelines
        self.icd10_mapping = {
            # Cardiovascular conditions
            "chest pain": "R07.9",
            "chest pain unspecified": "R07.9",
            "angina": "I20.9",
            "unstable angina": "I20.0",
            "stable angina": "I20.8",
            "hypertension": "I10",
            "essential hypertension": "I10",
            "hypertensive heart disease": "I11.9",
            "stemi": "I21.3",
            "st elevation myocardial infarction": "I21.3",
            "st-elevation myocardial infarction": "I21.3",
            "myocardial infarction": "I21.9",
            "acute myocardial infarction": "I21.9",
            "heart failure": "I50.9",
            "congestive heart failure": "I50.9",
            "chf": "I50.9",
            "systolic heart failure": "I50.21",
            "diastolic heart failure": "I50.31",
            "atrial fibrillation": "I48.91",
            "afib": "I48.91",
            "atrial flutter": "I48.92",
            "ventricular tachycardia": "I47.2",
            "ventricular fibrillation": "I49.01",
            "cardiac arrest": "I46.9",
            "cardiogenic shock": "R57.0",

            # Pulmonary conditions
            "pneumonia": "J18.9",
            "community-acquired pneumonia": "J18.9",
            "cap": "J18.9",
            "aspiration pneumonia": "J69.0",
            "copd": "J44.9",
            "chronic obstructive pulmonary disease": "J44.9",
            "asthma": "J45.909",
            "acute asthma exacerbation": "J45.901",
            "respiratory failure": "J96.90",
            "acute respiratory failure": "J96.00",
            "pulmonary embolism": "I26.99",
            "pe": "I26.99",
            "pulmonary edema": "J81.0",
            "pleural effusion": "J90",

            # Neurological conditions
            "stroke": "I63.9",
            "ischemic stroke": "I63.9",
            "cva": "I63.9",
            "cerebrovascular accident": "I63.9",
            "hemorrhagic stroke": "I61.9",
            "intracerebral hemorrhage": "I61.9",
            "subarachnoid hemorrhage": "I60.9",
            "transient ischemic attack": "G45.9",
            "tia": "G45.9",
            "seizure": "R56.9",
            "status epilepticus": "G40.901",
            "epilepsy": "G40.909",
            "meningitis": "G03.9",
            "encephalitis": "G04.90",

            # Endocrine conditions
            "diabetes": "E11.9",
            "diabetes mellitus": "E11.9",
            "t2dm": "E11.9",
            "type 2 diabetes": "E11.9",
            "type 2 diabetes mellitus": "E11.9",
            "type 1 diabetes": "E10.9",
            "t1dm": "E10.9",
            "diabetic ketoacidosis": "E11.10",
            "dka": "E11.10",
            "hypoglycemia": "E16.2",
            "hyperglycemia": "R73.9",
            "thyrotoxicosis": "E05.90",
            "hypothyroidism": "E03.9",

            # Infectious diseases
            "sepsis": "A41.9",
            "severe sepsis": "R65.20",
            "septic shock": "R65.21",
            "bacteremia": "A49.9",
            "urinary tract infection": "N39.0",
            "uti": "N39.0",
            "cellulitis": "L03.90",
            "influenza": "J11.1",
            "covid-19": "U07.1",

            # Renal conditions
            "acute kidney injury": "N17.9",
            "aki": "N17.9",
            "chronic kidney disease": "N18.9",
            "ckd": "N18.9",
            "end stage renal disease": "N18.6",
            "esrd": "N18.6",
            "renal failure": "N19",

            # Gastrointestinal conditions
            "gastrointestinal bleeding": "K92.2",
            "gi bleed": "K92.2",
            "upper gi bleeding": "K92.2",
            "lower gi bleeding": "K62.5",
            "cirrhosis": "K74.60",
            "hepatic failure": "K72.90",
            "ascites": "R18.8",
            "pancreatitis": "K85.90",
            "acute pancreatitis": "K85.90",
            "bowel obstruction": "K56.60",
            "ileus": "K56.7",

            # Hematologic conditions
            "anemia": "D64.9",
            "iron deficiency anemia": "D50.9",
            "thrombocytopenia": "D69.6",
            "deep vein thrombosis": "I82.40",
            "dvt": "I82.40",
            "disseminated intravascular coagulation": "D65",
            "dic": "D65",

            # Trauma/Emergency
            "shock": "R57.9",
            "hypovolemic shock": "R57.1",
            "hemorrhagic shock": "R57.1",
            "anaphylaxis": "T78.2",
            "syncope": "R55",
            "altered mental status": "R41.82",
        }

        # Critical condition patterns (Life-threatening emergencies requiring immediate intervention)
        # Based on: ACLS guidelines, Surviving Sepsis Campaign, AHA Stroke Guidelines
        self.critical_patterns = [
            # Cardiac emergencies (time-sensitive)
            (r'\b(STEMI|ST[- ]elevation|ST elevation myocardial infarction)\b',
             " STEMI - IMMEDIATE CARDIAC CATH (Door-to-balloon <90 min)"),
            (r'\b(cardiac arrest|ventricular fibrillation|v[- ]?fib|pulseless)\b',
             " CARDIAC ARREST - IMMEDIATE ACLS PROTOCOL"),
            (r'\b(cardiogenic shock|acute decompensated heart failure|ADHF)\b',
             " CARDIOGENIC SHOCK - ICU ADMISSION & INOTROPES"),
            (r'\b(unstable angina|crescendo angina)\b',
             " UNSTABLE ANGINA - CARDIOLOGY CONSULT URGENT"),

            # Neurological emergencies (time-sensitive)
            (r'\b(stroke|CVA|cerebrovascular accident|acute ischemic stroke)\b',
             " ACUTE STROKE - TIME-SENSITIVE tPA WINDOW (4.5hr)"),
            (r'\b(intracerebral hemorrhage|ICH|brain bleed|hemorrhagic stroke)\b',
             " HEMORRHAGIC STROKE - IMMEDIATE NEUROSURGERY CONSULT"),
            (r'\b(status epilepticus|continuous seizure)\b',
             " STATUS EPILEPTICUS - IMMEDIATE BENZODIAZEPINES"),
            (r'\b(subarachnoid hemorrhage|SAH)\b',
             " SUBARACHNOID HEMORRHAGE - NEUROSURGERY STAT"),

            # Infectious/Septic emergencies
            (r'\b(sepsis|septic shock|severe sepsis)\b',
             " SEPSIS - SEPSIS PROTOCOL (Antibiotics <1hr, Fluids)"),
            (r'\b(meningitis|bacterial meningitis)\b',
             " MENINGITIS - IMMEDIATE ANTIBIOTICS & LP"),

            # Respiratory emergencies
            (r'\b(respiratory failure|acute respiratory distress|ARDS)\b',
             " RESPIRATORY FAILURE - CONSIDER INTUBATION"),
            (r'\b(massive pulmonary embolism|massive PE)\b',
             " MASSIVE PE - THROMBOLYTICS VS EMBOLECTOMY"),
            (r'\b(tension pneumothorax)\b',
             " TENSION PNEUMOTHORAX - IMMEDIATE NEEDLE DECOMPRESSION"),

            # Metabolic emergencies
            (r'\b(DKA|diabetic ketoacidosis)\b',
             " DKA - INSULIN DRIP & AGGRESSIVE FLUIDS"),
            (r'\b(hyperosmolar hyperglycemic state|HHS)\b',
             " HHS - AGGRESSIVE FLUID RESUSCITATION"),

            # Trauma/Hemorrhage
            (r'\b(hemorrhagic shock|massive hemorrhage|active bleeding)\b',
             " HEMORRHAGIC SHOCK - MASSIVE TRANSFUSION PROTOCOL"),
            (r'\b(ruptured aneurysm|aortic dissection)\b',
             " AORTIC EMERGENCY - IMMEDIATE SURGERY CONSULT"),

            # Anaphylaxis
            (r'\b(anaphylaxis|anaphylactic shock)\b',
             " ANAPHYLAXIS - IMMEDIATE EPINEPHRINE IM"),
        ]

        print(f" ICD-10 Codes loaded: {len(self.icd10_mapping)} conditions")
        print(f" Critical patterns loaded: {len(self.critical_patterns)} emergency conditions")

    def extract_entities_rule_based(self, text):
        """Rule-based entity extraction (baseline)"""
        entities = {
            "Diagnoses": [],
            "Medications": [],
            "Symptoms": [],
            "Lab Results": []
        }

        # Diagnoses
        diag_pattern = r'\b(STEMI|MI|myocardial infarction|hypertension|diabetes|T2DM|pneumonia|COPD|heart failure|stroke|sepsis|cirrhosis|atrial fibrillation)\b'
        entities["Diagnoses"] = list(set(re.findall(diag_pattern, text, re.IGNORECASE)))

        # Medications
        med_pattern = r'\b([A-Z][a-z]+(?:ol|pril|formin|statin|mycin|cillin))\s*(\d+\s*mg)?(?:\s*(BID|QD|TID))?\b'
        meds = re.findall(med_pattern, text)
        entities["Medications"] = [f"{m[0]} {m[1]} {m[2]}".strip() for m in meds if m[0]]

        # Symptoms
        symp_pattern = r'\b(chest pain|shortness of breath|fever|cough|dyspnea|weakness|edema)\b'
        entities["Symptoms"] = list(set(re.findall(symp_pattern, text, re.IGNORECASE)))

        # Labs
        lab_pattern = r'\b(troponin|HbA1c|WBC|SpO2|BP|HR|BNP|creatinine|eGFR)\s*:?\s*([\d\.]+)\s*([a-zA-Z/%]+)?'
        labs = re.findall(lab_pattern, text, re.IGNORECASE)
        entities["Lab Results"] = [f"{l[0]}: {l[1]} {l[2]}".strip() for l in labs]

        return entities

    def extract_entities_model_based(self, text):
        """Model-based entity extraction using BioClinicalBERT"""
        if not self.models_loaded:
            return self.extract_entities_rule_based(text)

        entities = {
            "Diagnoses": [],
            "Medications": [],
            "Symptoms": [],
            "Procedures": [],
            "Confidence": []
        }

        try:
            # Run NER pipeline (aggregation_strategy="simple" already merges subwords)
            ner_results = ner_pipeline(text)

            # NEW: Entity Merging Logic - Combine adjacent fragments of same entity type
            # This fixes subword tokenization issues like "diclofenac" → "di" + "cl" + "ofenac"
            ner_results = sorted(ner_results, key=lambda x: x.get('start', 0))

            merged_results = []
            i = 0
            while i < len(ner_results):
                current_entity = ner_results[i]
                merged_word = current_entity.get('word', '').strip()
                merged_start = current_entity.get('start', 0)
                merged_end = current_entity.get('end', 0)
                entity_type = current_entity.get('entity_group', current_entity.get('entity', 'Unknown'))
                scores = [current_entity['score']]

                # Check if next entities are adjacent and same type
                j = i + 1
                while j < len(ner_results):
                    next_entity = ner_results[j]
                    next_start = next_entity.get('start', 0)
                    next_type = next_entity.get('entity_group', next_entity.get('entity', 'Unknown'))

                    # If next entity is adjacent (within 2 chars for spacing) and same type, merge
                    if next_start - merged_end <= 2 and next_type == entity_type:
                        # Extract text between current and next entity from original text
                        gap_text = text[merged_end:next_start] if merged_end < next_start else ''
                        next_word = next_entity.get('word', '').strip()
                        merged_word = f"{merged_word}{gap_text}{next_word}"
                        merged_end = next_entity.get('end', merged_end)
                        scores.append(next_entity['score'])
                        j += 1
                    else:
                        break

                # Create merged entity with averaged confidence
                merged_entity = {
                    'word': merged_word,
                    'entity_group': entity_type,
                    'entity': entity_type,
                    'score': sum(scores) / len(scores),
                    'start': merged_start,
                    'end': merged_end
                }
                merged_results.append(merged_entity)
                i = j

            # Use merged results for further processing
            ner_results = merged_results

            # Debug: Track entity types seen
            entity_types_seen = set()

            # Process entities
            for entity in ner_results:
                # Get entity information
                entity_text = entity.get('word', '').strip()
                entity_score = entity['score']
                entity_type = entity.get('entity_group', entity.get('entity', 'Unknown'))

                # The aggregation_strategy="simple" should already merge subwords,
                # but clean up any remaining ## markers
                entity_text = entity_text.replace('##', '').replace(' ##', '').strip()

                # Skip very short tokens (likely fragments or noise)
                if len(entity_text) < 3:
                    continue

                # Track what entity types we're seeing
                entity_types_seen.add(entity_type)

                # Map BioClinicalBERT's entity types to our categories
                entity_type_upper = entity_type.upper()

                # Diagnoses: diseases, disorders, conditions, problems
                if any(term in entity_type_upper for term in [
                    'DISEASE', 'DISORDER', 'CONDITION', 'PROBLEM',
                    'DISEASE_DISORDER', 'INJURY_OR_POISONING'
                ]):
                    entities["Diagnoses"].append(f"{entity_text} ({entity_score:.2f})")

                # Medications: chemicals, drugs, treatments
                elif any(term in entity_type_upper for term in [
                    'CHEMICAL', 'DRUG', 'MEDICATION', 'TREATMENT',
                    'PHARMACOLOGIC_SUBSTANCE', 'SUBSTANCE'
                ]):
                    entities["Medications"].append(f"{entity_text} ({entity_score:.2f})")

                # Symptoms: signs, findings, observations
                elif any(term in entity_type_upper for term in [
                    'SYMPTOM', 'SIGN', 'FINDING', 'OBSERVATION',
                    'SIGN_OR_SYMPTOM', 'CLINICAL_ATTRIBUTE'
                ]):
                    entities["Symptoms"].append(f"{entity_text} ({entity_score:.2f})")

                # Procedures: tests, exams, therapeutic procedures
                elif any(term in entity_type_upper for term in [
                    'PROCEDURE', 'TEST', 'EXAM', 'DIAGNOSTIC',
                    'THERAPEUTIC_OR_PREVENTIVE_PROCEDURE', 'LABORATORY'
                ]):
                    entities["Procedures"].append(f"{entity_text} ({entity_score:.2f})")

                # Additional categories for d4data/biomedical-ner-all specific labels
                elif 'BIOLOGICAL_STRUCTURE' in entity_type_upper or 'ANATOMY' in entity_type_upper:
                    # Anatomical structures -> categorize as context for symptoms
                    entities["Symptoms"].append(f"{entity_text} ({entity_score:.2f})")

                elif 'CLINICAL_EVENT' in entity_type_upper:
                    # Clinical events -> procedures
                    entities["Procedures"].append(f"{entity_text} ({entity_score:.2f})")

                # Skip demographic/administrative entities (Age, Sex, Location, etc.)
                elif any(skip_term in entity_type_upper for skip_term in [
                    'AGE', 'SEX', 'GENDER', 'LOCATION', 'DATE', 'TIME',
                    'SEVERITY', 'DETAILED_DESCRIPTION', 'NONBIOLOGICAL'
                ]):
                    continue

                entities["Confidence"].append(entity_score)

            # Debug output
            print(f"DEBUG: Entity types found: {entity_types_seen}")
            print(f"DEBUG: Extracted - Diagnoses: {len(entities['Diagnoses'])}, Medications: {len(entities['Medications'])}, Symptoms: {len(entities['Symptoms'])}, Procedures: {len(entities['Procedures'])}")

            # Calculate average confidence
            if entities["Confidence"]:
                avg_conf = sum(entities["Confidence"]) / len(entities["Confidence"])
                entities["Average_Confidence"] = f"{avg_conf:.2%}"

        except Exception as e:
            print(f"Model inference failed: {e}")
            return self.extract_entities_rule_based(text)

        return entities

    def generate_summary_model(self, text):
        """Generate summary using BART model"""
        if not self.models_loaded or len(text) < 100:
            return self.generate_summary_rule_based(text)

        try:
            # Check if summarizer is available
            if summarizer is None:
                return self.generate_summary_rule_based(text)

            # Call BART summarization function
            summary = summarizer(text, max_length=130, min_length=30)
            return summary
        except Exception as e:
            print(f"Summarization failed: {e}")
            return self.generate_summary_rule_based(text)

    def generate_summary_rule_based(self, text):
        """Rule-based summarization (fallback)"""
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        keywords = ['impression', 'plan', 'diagnosis']

        key_sentences = []
        for sent in sentences[:10]:
            if any(kw in sent.lower() for kw in keywords):
                key_sentences.append(sent)

        return ". ".join(key_sentences[:2]) + "." if key_sentences else sentences[0] + "."

    def map_to_icd10(self, diagnoses):
        """
        Map diagnoses to ICD-10-CM codes using comprehensive medical coding

        Uses two-tier approach:
        1. Direct mapping from comprehensive dictionary (100+ conditions)
        2. Fuzzy matching with simple-icd-10 library if available
        """
        results = []

        for diag in diagnoses:
            # Remove confidence scores if present
            diag_clean = re.sub(r'\s*\([\d\.]+\)', '', diag).lower().strip()

            # Tier 1: Direct mapping from comprehensive dictionary
            if diag_clean in self.icd10_mapping:
                code = self.icd10_mapping[diag_clean]
                results.append((diag, code, "Direct mapping"))
                continue

            # Tier 2: Fuzzy matching with ICD-10 library (if available)
            if self.icd10_available:
                try:
                    # Use simple_icd_10 for fuzzy search
                    import simple_icd_10 as icd

                    # Try to find matching code
                    search_results = icd.search(diag_clean)

                    if search_results:
                        # Take first match (most relevant)
                        best_match = search_results[0]
                        code = best_match['code']
                        description = best_match['description']
                        results.append((diag, code, f"Fuzzy match: {description[:50]}"))
                        continue

                except Exception as e:
                    # Fallback if fuzzy matching fails
                    pass

            # No match found - indicate unmapped
            results.append((diag, "UNMAPPED", "No ICD-10 code found"))

        return results

    def detect_critical(self, text):
        """Detect critical conditions"""
        alerts = []
        for pattern, desc in self.critical_patterns:
            if re.search(pattern, text, re.IGNORECASE):
                alerts.append(desc)
        return list(set(alerts))

    def analyze(self, text, use_models=True):
        """Complete analysis with model comparison"""
        if not text or len(text.strip()) < 10:
            return {"error": "Please enter valid clinical note"}

        results = {}

        # Model-based extraction (primary)
        if use_models and self.models_loaded:
            results['entities_model'] = self.extract_entities_model_based(text)
            results['summary_model'] = self.generate_summary_model(text)
            results['method'] = "BioClinicalBERT + BART"
        else:
            results['entities_model'] = self.extract_entities_rule_based(text)
            results['summary_model'] = self.generate_summary_rule_based(text)
            results['method'] = "Rule-based"

        # Rule-based extraction (for comparison)
        results['entities_rules'] = self.extract_entities_rule_based(text)
        results['summary_rules'] = self.generate_summary_rule_based(text)

        # ICD-10 codes
        diag_list = results['entities_model'].get('Diagnoses', [])
        results['icd10_codes'] = self.map_to_icd10(diag_list)

        # Critical alerts
        results['critical_alerts'] = self.detect_critical(text)

        return results

# Initialize enhanced analyzer
analyzer = MedicalNLPAnalyzer(device=device, models_loaded=models_loaded)
print("Medical NLP Analyzer initialized!")

======================================================================
Step 6: Initializing Medical NLP Analyzer System...
======================================================================

 ICD-10 Codes loaded: 106 conditions
 Critical patterns loaded: 18 emergency conditions
Medical NLP Analyzer initialized!

print("\n" + "=" * 70)
print("Step 7: Building Enhanced Gradio Interface...")
print("=" * 70 + "\n")

def analyze_note_enhanced(text, use_models, show_comparison):
    """Enhanced Gradio interface with model comparison"""

    results = analyzer.analyze(text, use_models=use_models)

    if "error" in results:
        return results["error"]

    output = ""

    # Critical alerts
    if results.get('critical_alerts'):
        output += "### CRITICAL ALERTS\n\n"
        for alert in results['critical_alerts']:
            output += f"**{alert}**\n\n"
        output += "---\n\n"

    # Model info
    output += f"### Analysis Method: **{results['method']}**\n\n"

    # Summary (Model-based)
    output += f"### Clinical Summary\n\n{results['summary_model']}\n\n"

    if show_comparison and use_models and models_loaded:
        output += f"**Rule-based summary:** {results['summary_rules']}\n\n"

    output += "---\n\n"

    # Entities (Model-based)
    output += "### Extracted Medical Entities (Model-Based)\n\n"
    for entity_type, entities in results['entities_model'].items():
        if entity_type not in ['Confidence', 'Average_Confidence'] and entities:
            output += f"**{entity_type}** ({len(entities)}):\n"
            for entity in entities[:10]:  # Limit to 10
                output += f"  • {entity}\n"
            output += "\n"

    # Show average confidence if available
    if 'Average_Confidence' in results['entities_model']:
        output += f"**Model Confidence:** {results['entities_model']['Average_Confidence']}\n\n"

    # Comparison with rule-based
    if show_comparison:
        output += "###   Comparison: Rule-Based vs Model-Based\n\n"
        output += f"**Rule-based found:** {sum(len(v) for v in results['entities_rules'].values())} entities\n"
        output += f"**Model-based found:** {sum(len(v) for k, v in results['entities_model'].items() if k not in ['Confidence', 'Average_Confidence'])} entities\n\n"

    output += "---\n\n"

    # ICD-10 codes with match type
    if results.get('icd10_codes'):
        output += "###  ICD-10-CM Medical Codes\n\n"
        output += "| Diagnosis | Code | Match Type |\n"
        output += "|-----------|------|------------|\n"

        for item in results['icd10_codes']:
            if len(item) == 3:
                diag, code, match_type = item
                output += f"| {diag} | **{code}** | {match_type} |\n"
            else:
                # Fallback for backward compatibility
                diag, code = item[0], item[1]
                output += f"| {diag} | **{code}** | Direct |\n"

        output += "\n"
        output += f"**Coverage:** {len(analyzer.icd10_mapping)} conditions in direct mapping database\n\n"

    return output

print(" Enhanced Gradio interface ready!")

======================================================================
Step 7: Building Enhanced Gradio Interface...
======================================================================

 Enhanced Gradio interface ready!

print("\n" + "=" * 70)
print("Step 8: Launching Medical NLP Analyzer Demo...")
print("=" * 70 + "\n")

# Build Gradio interface
with gr.Blocks(title="Medical NLP Analyzer") as demo:

    gr.Markdown("#  Medical NLP Analyzer")
    gr.Markdown("**With Real LLMs, BioClinicalBERT, and Public Medical Dataset**")
    gr.Markdown("Built by Abhishek Kumar Singh")
    gr.Markdown("---")

    gr.Markdown("""
    ### Key Features:
    - **Real Models**: BioClinicalBERT for medical NER
    - **Public Data**: Augmented_clinical_notes_dataset medical transcriptions
    - **GPU Accelerated**: T4 GPU for fast inference
    - **Model Comparison**: See rule-based vs ML-based results
    """)

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Input")

            # Dataset selector
            dataset_dropdown = gr.Dropdown(
                choices=[f"{i+1}. {d['specialty']}" for i, d in enumerate(MEDICAL_DATASET)],
                label="Load from Augmented Clinical Notes Dataset:",
                value=f"1. {MEDICAL_DATASET[0]['specialty']}"
            )

            text_input = gr.Textbox(
                label="Clinical Note:",
                lines=12,
                value=MEDICAL_DATASET[0]['note']
            )

            def load_from_dataset(selection):
                idx = int(selection.split('.')[0]) - 1
                return MEDICAL_DATASET[idx]['note']

            dataset_dropdown.change(
                fn=load_from_dataset,
                inputs=[dataset_dropdown],
                outputs=[text_input]
            )

            # Options
            use_models_checkbox = gr.Checkbox(
                label=" Use Medical LLMs (BioClinicalBERT + BART)",
                value=models_loaded
            )

            show_comparison_checkbox = gr.Checkbox(
                label=" Show Rule-based vs Model Comparison",
                value=True
            )

            analyze_btn = gr.Button(" Analyze with Medical LLMs", variant="primary")

        with gr.Column():
            gr.Markdown("### Results")
            output = gr.Markdown()

    analyze_btn.click(
        fn=analyze_note_enhanced,
        inputs=[text_input, use_models_checkbox, show_comparison_checkbox],
        outputs=[output]
    )

    gr.Markdown("---")
    gr.Markdown("""
    ### About This Demo:
    - **Uses BioClinicalBERT**: Medical entity recognition model
    - **BART Summarization**: Clinical note summarization
    - **Augmented_clinical_notes_dataset**: Real medical transcriptions
    - **GPU Acceleration**: Faster inference on T4
    - **Model Confidence**: Shows prediction confidence scores

    *Synthetic data for demonstration. Not for clinical use.*
    """)

print(" Launching enhanced demo with public URL...")
demo.launch(share=True, debug=False)

print("\n Medical NLP Demo launched!")
print("\n This demonstrates:")
print(" Real medical LLMs (BioClinicalBERT, BART)")
print(" Public medical dataset (Augmented Clinical Dataset)")
print(" GPU-accelerated inference")
print(" Model confidence scores")
print(" Comparison with rule-based baseline")
print("\n" + "=" * 70)

Colab paid products - Cancel contracts here


In [ ]:
logger.info("\n" + "="*80)
logger.info("GENERATING FINAL CONSOLIDATED REPORT & PPT ASSETS")
logger.info("="*80)

# Run a sample analysis to trigger all visualizations
sample_note = examples[0][0]
logger.info(f"Running automated report generation on sample note...")

# This call triggers the "Entity Type Distribution" plot saving inside the function
report_results = analyze_clinical_note(sample_note)

# --- Generate NER Confidence Histogram (for PPT) ---
if report_results["entity_confidence_distribution"]:
    try:
        confidences = [c for c in report_results["entity_confidence_distribution"] if 0 < c <= 1.0]

        plt.figure(figsize=(10, 6))
        plt.hist(confidences, bins=20, color='#50c878', edgecolor='black', range=(0.0, 1.0))
        plt.title('NER Model Confidence Distribution', fontsize=14, fontweight='bold')
        plt.xlabel('Confidence Score (Probability)', fontsize=12)
        plt.ylabel('Number of Tokens', fontsize=12)
        plt.grid(axis='y', alpha=0.3)
        plt.axvline(config["ner_confidence_threshold"], color='red', linestyle='dashed', linewidth=1, label=f'Threshold ({config["ner_confidence_threshold"]})')
        plt.legend()
        plt.tight_layout()

        plot_path = os.path.join(config["plot_output_dir"], "ner_confidence_distribution.png")
        plt.savefig(plot_path, dpi=300)
        logger.info(f"Saved NER confidence distribution plot to {plot_path}")
        plt.close()
    except Exception as e:
        logger.error(f"Failed to plot confidence distribution: {e}")

# Save the Text Report
report_md_path = os.path.join(config["summary_output_dir"], "final_analysis_report.md")
try:
    formatted_report = format_output_for_gradio(sample_note)
    with open(report_md_path, "w") as f:
        f.write(formatted_report)
    logger.info(f"Saved full markdown report to {report_md_path}")
except Exception as e:
    logger.error(f"Failed to save text report: {e}")

logger.info("\n DEMO EXECUTION COMPLETE.")
logger.info(f"Artifacts available in: {config['plot_output_dir']}")


ERROR:__main__:Error during NER: name 'text' is not defined
ERROR:__main__:Error during ICD-10 coding: cannot access local variable 'extracted_entities' where it is not associated with a value
ERROR:__main__:Error during NER: name 'text' is not defined
ERROR:__main__:Error during ICD-10 coding: cannot access local variable 'extracted_entities' where it is not associated with a value
